# SIB - LogisticRegression (Gradient Descent + L2)

Teste da classe `LogisticRegression` com o dataset `breast-bin.csv` (classificação binária).

O modelo aplica a função sigmoide à combinação linear das features, obtendo a probabilidade de a amostra pertencer à classe 1:

$$h_\theta(x) = \sigma(\theta^T x) = \frac{1}{1 + e^{-\theta^T x}}$$

e estima os coeficientes por **gradient descent** com regularização **L2**, minimizando a cross-entropy regularizada:

$$J(\theta) = -\frac{1}{m} \sum_{i=1}^{m} \left[ y^{(i)} \log h_\theta(x^{(i)}) + (1 - y^{(i)}) \log (1 - h_\theta(x^{(i)})) \right] + \frac{\lambda}{2m} \sum_{j=1}^{n} \theta_j^2$$

As extensões (momentum e L1) estão em `extra_logistic_regression.ipynb`.

In [1]:
import numpy as np
import matplotlib.pyplot as plt

from si.io.csv_file import read_csv
from si.metrics.accuracy import accuracy
from si.model_selection.split import stratified_train_test_split
from si.models.logistic_regression import LogisticRegression
from si.statistics.sigmoid_function import sigmoid_function

## 1. sigmoid_function

Mapeia qualquer valor real para o intervalo ]0, 1[; $\sigma(0) = 0.5$ é a fronteira de decisão.

In [2]:
z = np.array([-10.0, -1.0, 0.0, 1.0, 10.0])
print('z:         ', z)
print('sigmoid(z):', np.round(sigmoid_function(z), 4))

z:          [-10.  -1.   0.   1.  10.]
sigmoid(z): [0.     0.2689 0.5    0.7311 1.    ]


## 2. Dataset breast-bin.csv

Tumor benigno (0) vs maligno (1). O ficheiro não tem cabeçalho, por isso `features=False`.

In [3]:
breast = read_csv('../datasets/breast_bin/breast-bin.csv', sep=',', features=False, label=True)

print('dimensões:', breast.shape())
classes, counts = np.unique(breast.y, return_counts=True)
for c, n in zip(classes, counts):
    print('  classe {:.0f}: {:>4d} amostras ({:.1%})'.format(c, n, n / len(breast.y)))

dimensões: (698, 9)
  classe 0:  458 amostras (65.6%)
  classe 1:  240 amostras (34.4%)


## 3. Divisão em treino e teste

As classes estão desequilibradas (~65/35), por isso uso a divisão estratificada: treino e teste ficam com a mesma proporção de classes.

In [4]:
train, test = stratified_train_test_split(breast, test_size=0.2, random_state=42)

print('treino:', train.shape(), '| proporção classe 1: {:.3f}'.format(train.y.mean()))
print('teste: ', test.shape(), '| proporção classe 1: {:.3f}'.format(test.y.mean()))

treino: (559, 9) | proporção classe 1: 0.343
teste:  (139, 9) | proporção classe 1: 0.345


## 4. Treino do modelo

In [5]:
model = LogisticRegression(l2_penalty=1.0, alpha=0.1, max_iter=2000, patience=20, scale=True)
model.fit(train)

print('iterações efetuadas:', len(model.cost_history), '(max_iter = 2000)')
print('theta_zero: {:.4f}'.format(model.theta_zero))
for j, coef in enumerate(model.theta):
    print('  x{:<2d} {:>9.4f}'.format(j, coef))

iterações efetuadas: 172 (max_iter = 2000)
theta_zero: -0.6558
  x0     0.6935
  x1     0.5657
  x2     0.6326
  x3     0.5403
  x4     0.4058
  x5     0.8667
  x6     0.6316
  x7     0.4855
  x8     0.3399


## 5. Score e custo

`score` devolve a **accuracy**; `cost` devolve a função de custo J (cross-entropy + termo L2).
A accuracy só tem significado comparada com o classificador trivial que prevê sempre a classe maioritária.

In [6]:
print('accuracy (treino): {:.4f}'.format(model.score(train)))
print('accuracy (teste):  {:.4f}'.format(model.score(test)))
print('cost     (treino): {:.4f}'.format(model.cost(train)))
print('cost     (teste):  {:.4f}'.format(model.cost(test)))

majority = np.zeros_like(test.y)
print()
print('baseline (prevê sempre 0), accuracy teste: {:.4f}'.format(accuracy(test.y, majority)))

accuracy (treino): 0.9678
accuracy (teste):  0.9784
cost     (treino): 0.1024
cost     (teste):  0.0977

baseline (prevê sempre 0), accuracy teste: 0.6547


In [7]:
y_pred = model.predict(test)
tp = int(np.sum((y_pred == 1) & (test.y == 1)))
tn = int(np.sum((y_pred == 0) & (test.y == 0)))
fp = int(np.sum((y_pred == 1) & (test.y == 0)))
fn = int(np.sum((y_pred == 0) & (test.y == 1)))

print('matriz de confusão (teste)')
print('              prev. 0  prev. 1')
print('  real 0     {:>7d}  {:>7d}'.format(tn, fp))
print('  real 1     {:>7d}  {:>7d}'.format(fn, tp))

matriz de confusão (teste)
              prev. 0  prev. 1
  real 0          89        2
  real 1           1       47


## 6. Convergência

In [8]:
costs = list(model.cost_history.values())

plt.figure(figsize=(6, 3.5))
plt.plot(costs)
plt.xlabel('iteração')
plt.ylabel('custo J')
plt.title('Função de custo (treino)')
plt.tight_layout()
plt.show()

print('custo inicial: {:.4f}'.format(costs[0]))
print('custo final:   {:.4f}'.format(costs[-1]))

custo inicial: 0.5920
custo final:   0.1024


## 7. Efeito do l2_penalty

In [9]:
print('{:>12s} {:>12s} {:>12s} {:>14s}'.format('l2_penalty', 'sum|theta|', 'acc. teste', 'custo teste'))
for l2 in [0.0, 1.0, 10.0, 100.0, 1000.0]:
    m = LogisticRegression(l2_penalty=l2, alpha=0.1, max_iter=2000, patience=20, scale=True).fit(train)
    print('{:>12.1f} {:>12.4f} {:>12.4f} {:>14.4f}'.format(
        l2, np.sum(np.abs(m.theta)), m.score(test), m.cost(test)))

  l2_penalty   sum|theta|   acc. teste    custo teste
         0.0       5.2636       0.9784         0.0854
         1.0       5.1616       0.9784         0.0977
        10.0       4.5359       0.9784         0.1816
       100.0       2.7598       0.9784         0.4739
      1000.0       0.9764       0.9353         0.7586


## Conclusões

- **Score**: accuracy de 0.978 no teste (0.968 no treino), contra 0.655 do classificador que prevê sempre "benigno". No teste há 3 erros em 139 amostras: 2 falsos positivos e 1 falso negativo.
- **Custo**: J desce de 0.592 para 0.102 no treino em 172 iterações (paragem por `patience`); no teste J = 0.098. O custo de teste ser ligeiramente inferior ao de treino indica que não há overfitting, mas com 139 amostras de teste esta diferença está dentro do ruído.
- **L2**: aumentar o `l2_penalty` encolhe os coeficientes e faz subir o custo, mas a accuracy mantém-se até lambda = 100. A accuracy é pouco sensível porque só depende de que lado de 0.5 cai a probabilidade; o custo mede também a confiança das previsões. Só com lambda = 1000 o modelo fica demasiado enviesado e a accuracy cai.
- Todos os coeficientes são positivos: cada feature (escalada) aumenta a probabilidade de malignidade, o que é coerente com o dataset (valores mais altos = células mais anormais).